# Data Cleaning — SAP News Intelligence

**Inputs** (all in `notebook/data/`):
| File | Source | Rows | Notes |
|---|---|---|---|
| `sap_news.json` | SAP News RSS + scraper | 150 | Full article content available |
| `gnews_articles.json` | GNews API | 43 | Content truncated by API; use description |
| `hackernews_posts.json` | Hacker News | 220 | 107/220 rows have article content |

**Output:** `notebook/data/clean_data.json`


## 1. Imports & configuration

In [ ]:
import re
import unicodedata
from pathlib import Path

import pandas as pd
from rapidfuzz import fuzz

# ── Resolve notebook/data/ regardless of where Jupyter sets CWD ─────────────
_cwd = Path.cwd()
candidates = [
    _cwd / "data",           # running from notebook/
    _cwd / "notebook" / "data",  # running from repo root
    _cwd.parent / "data",    # fallback
]
DATA_DIR = next((p for p in candidates if p.exists()), candidates[0])

SAP_PATH   = DATA_DIR / "sap_news.json"
GNEWS_PATH = DATA_DIR / "gnews_articles.json"
HN_PATH    = DATA_DIR / "hackernews_posts.json"
OUT_PATH   = DATA_DIR / "clean_data.json"

MIN_TEXT_LEN    = 100   # drop articles shorter than this after cleaning
FUZZY_THRESHOLD = 85    # deduplicate near-identical titles

print(f"Working dir : {_cwd}")
print(f"Data dir    : {DATA_DIR}  (exists: {DATA_DIR.exists()})")
for p in [SAP_PATH, GNEWS_PATH, HN_PATH]:
    print(f"  {p.name}: {p.exists()}")


## 2. Load raw sources

In [ ]:
sap_df   = pd.read_json(SAP_PATH)
gnews_df = pd.read_json(GNEWS_PATH)
hn_df    = pd.read_json(HN_PATH)

print(f"SAP News     : {sap_df.shape}   columns: {sap_df.columns.tolist()}")
print(f"GNews        : {gnews_df.shape}  columns: {gnews_df.columns.tolist()}")
print(f"Hacker News  : {hn_df.shape}  columns: {hn_df.columns.tolist()}")


## 3. Build `text` field per source

Each source needs different handling:
- **SAP News**: full content available — use `title + description + content`
- **GNews**: content is truncated by the API to ~266 chars — use `title + description + cleaned content`
- **Hacker News**: 107/220 rows have content; rest are titles only — use whatever is available


In [ ]:
COLS = ["title", "description", "content", "source", "published", "url"]

# SAP: all columns already present and aligned
sap_clean = sap_df[COLS].copy()
sap_clean["text"] = (
    sap_clean["title"].fillna("") + " "
    + sap_clean["description"].fillna("") + " "
    + sap_clean["content"].fillna("")
)

# GNews: content has API truncation markers — strip them, then combine
gnews_clean = gnews_df[COLS].copy()
gnews_clean["content_stripped"] = gnews_clean["content"].str.replace(
    r"\[\+\d+\s*chars\]", "", regex=True
).str.strip()
gnews_clean["text"] = (
    gnews_clean["title"].fillna("") + " "
    + gnews_clean["description"].fillna("") + " "
    + gnews_clean["content_stripped"].fillna("")
)
gnews_clean = gnews_clean.drop(columns=["content_stripped"])

# HackerNews: description is always empty; use title + content where available
hn_clean = hn_df[COLS].copy()
hn_clean["text"] = (
    hn_clean["title"].fillna("") + " "
    + hn_clean["content"].fillna("")
)

print(f"SAP text length   — mean: {sap_clean['text'].str.len().mean():.0f}, min: {sap_clean['text'].str.len().min()}")
print(f"GNews text length — mean: {gnews_clean['text'].str.len().mean():.0f}, min: {gnews_clean['text'].str.len().min()}")
print(f"HN text length    — mean: {hn_clean['text'].str.len().mean():.0f}, min: {hn_clean['text'].str.len().min()}")


## 4. Merge all sources

In [ ]:
all_docs = pd.concat(
    [sap_clean, gnews_clean, hn_clean],
    ignore_index=True
)

print(f"Combined shape: {all_docs.shape}")
print(f"Sources:\n{all_docs['source'].value_counts().to_string()}")


## 5. Clean text

Applies to the `text` field:
- Normalize unicode (NFKC: curly quotes → straight, ligatures, etc.)
- Remove invisible characters (`\u200b`, `\xa0`, BOM, etc.)
- Strip stray HTML tags
- Remove URLs and email addresses
- Remove GNews truncation markers (`[+N chars]`)
- Remove repeated separators (`***`, `---`, `===`, `>>>`)
- Collapse excess whitespace


In [ ]:
def clean_text(text: str) -> str:
    text = str(text)
    # Canonical unicode
    text = unicodedata.normalize("NFKC", text)
    # Invisible / control chars
    text = re.sub(r"[\u200b\u200c\u200d\u200e\u200f\u2060\ufeff\u2028\u2029]", " ", text)
    text = text.replace("\xa0", " ")
    # HTML tags
    text = re.sub(r"<[^>]+>", " ", text)
    # URLs
    text = re.sub(r"https?://\S+", "", text)
    # Emails
    text = re.sub(r"[\w.+-]+@[\w.-]+\.\w+", "", text)
    # GNews truncation markers
    text = re.sub(r"\[\+\d+\s*chars\]", "", text)
    # Repeated separators
    text = re.sub(r"[*]{3,}|[>]{3,}|[-]{3,}|[=]{3,}|[#]{3,}", " ", text)
    # Collapse whitespace
    text = re.sub(r"\s+", " ", text)
    return text.strip()


all_docs["clean_text"]   = all_docs["text"].apply(clean_text)
all_docs["clean_length"] = all_docs["clean_text"].str.len()

print(all_docs["clean_length"].describe())


## 6. Parse & sort dates

In [ ]:
all_docs["published"] = pd.to_datetime(all_docs["published"], errors="coerce", utc=True)

n_null = all_docs["published"].isna().sum()
print(f"Unparseable dates : {n_null} (kept as NaT)")
print(f"Date range        : {all_docs['published'].min()}  →  {all_docs['published'].max()}")

all_docs = all_docs.sort_values("published", na_position="last").reset_index(drop=True)


## 7. Filter irrelevant articles

Remove articles clearly unrelated to SAP / enterprise software.


In [ ]:
IRRELEVANT_TITLES = [
    "What is dragon's blood resin? The forgotten 2,000-year-old skincare ingredient used by ancient Roman and Arab women",
    "Tech updates (June 11, 2026): New ASUS gaming laptops, Canva Offline, Samsung AI deals",
    "Business wisdom of the day: 'The bamboo that bends is stronger than the oak that resists'",
    "India emerges as world's top marketing",
]

before = len(all_docs)
all_docs = all_docs[~all_docs["title"].isin(IRRELEVANT_TITLES)].reset_index(drop=True)
print(f"Removed {before - len(all_docs)} irrelevant articles")


## 8. Drop short articles

In [ ]:
before = len(all_docs)
all_docs = all_docs[all_docs["clean_length"] >= MIN_TEXT_LEN].reset_index(drop=True)
print(f"Dropped {before - len(all_docs)} articles shorter than {MIN_TEXT_LEN} chars  ({before} → {len(all_docs)})")


## 9. Deduplicate

Two passes:
1. **Exact** — same title
2. **Fuzzy** — titles with similarity ≥ 85 (catches rephrased duplicates)


In [ ]:
before = len(all_docs)

# Exact
all_docs = all_docs.drop_duplicates(subset=["title"]).reset_index(drop=True)
print(f"After exact dedup : {len(all_docs)}  (removed {before - len(all_docs)})")

# Fuzzy
titles = all_docs["title"].fillna("").tolist()
to_drop = set()
for i in range(len(titles)):
    if i in to_drop:
        continue
    for j in range(i + 1, len(titles)):
        if fuzz.ratio(titles[i], titles[j]) >= FUZZY_THRESHOLD:
            to_drop.add(j)

all_docs = all_docs.drop(index=list(to_drop)).reset_index(drop=True)
print(f"After fuzzy dedup : {len(all_docs)}  (removed {len(to_drop)} near-duplicates)")


## 10. Add `data_source_type` column

In [ ]:
all_docs["data_source_type"] = all_docs["source"].apply(
    lambda s: "owned" if s == "SAP News" else "third_party"
)
print(all_docs["data_source_type"].value_counts().to_string())


## 11. Final summary

In [ ]:
print("=" * 50)
print("  DATA CLEANING SUMMARY")
print("=" * 50)
print(f"  Total articles      : {len(all_docs)}")
print(f"  Columns             : {all_docs.columns.tolist()}")
print(f"  SAP News (owned)    : {(all_docs['data_source_type']=='owned').sum()}")
print(f"  Third-party         : {(all_docs['data_source_type']=='third_party').sum()}")
print(f"  Missing dates (NaT) : {all_docs['published'].isna().sum()}")
print(f"  Min clean_length    : {all_docs['clean_length'].min()}")
print(f"  Max clean_length    : {all_docs['clean_length'].max()}")
print(f"  Mean clean_length   : {all_docs['clean_length'].mean():.0f}")
print("=" * 50)
all_docs[["title","source","published","clean_length","data_source_type"]].head(5)


## 12. Save `clean_data.json`

In [ ]:
all_docs.to_json(OUT_PATH, orient="records", indent=4, date_format="iso")
print(f"Saved → {OUT_PATH}")
print(f"Shape  : {all_docs.shape}")
